# Boletim de Clima + Satélite REDEMET + n8n

Notebook organizado para:

1. Coletar clima de Pelotas pela Embrapa.
2. Coletar clima de Morro Redondo pela Open-Meteo.
3. Abrir a REDEMET, centralizar o mapa por latitude/longitude usando feedback da própria tela.
4. Ativar camada de satélite.
5. Capturar screenshot.
6. Enviar dados + imagem para o n8n.

> Observação: esta versão evita `ActionChains` para arrastar o mapa e usa o Chrome DevTools Protocol, reduzindo erros como `move target out of bounds`.

## 1. Bibliotecas e configurações

In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from datetime import datetime
import requests
import time
import re

# URL do webhook do n8n.
# Em modo teste do n8n geralmente é /webhook-test/...
# Em produção geralmente é /webhook/...
URL_WEBHOOK_N8N = "http://localhost:5678/webhook-test/boletim-clima"

# Defina como True se quiser fechar o navegador ao final.
FECHAR_NAVEGADOR_NO_FINAL = False

# Coordenadas alvo: centro aproximado entre Pelotas e Morro Redondo.
LAT_ALVO = -31.67
LON_ALVO = -52.48

# Janela fixa para deixar os cálculos e prints mais previsíveis.
LARGURA_JANELA = 1366
ALTURA_JANELA = 900

## 2. Configuração do Chrome

In [9]:
def configurar_navegador():
    opcoes = Options()
    opcoes.add_argument("--ignore-certificate-errors")
    opcoes.add_argument("--disable-blink-features=AutomationControlled")
    opcoes.add_argument(f"--window-size={LARGURA_JANELA},{ALTURA_JANELA}")

    driver = webdriver.Chrome(options=opcoes)
    driver.set_window_size(LARGURA_JANELA, ALTURA_JANELA)
    return driver

## 3. Coleta dos dados de Pelotas pela Embrapa

In [10]:
def acessar_frame_dados(driver):
    """Tenta encontrar o frame/iframe que contém os labels de clima da Embrapa."""
    driver.switch_to.default_content()

    frames = driver.find_elements(By.TAG_NAME, "frame") + driver.find_elements(By.TAG_NAME, "iframe")

    for indice in range(len(frames)):
        driver.switch_to.default_content()

        try:
            driver.switch_to.frame(indice)
            wait_curto = WebDriverWait(driver, 3)
            wait_curto.until(EC.presence_of_element_located((By.ID, "lblRealV1")))
            return True
        except Exception:
            continue

    driver.switch_to.default_content()
    return False


def extrair_dados_clima_pelotas(driver):
    """Abre o site da Embrapa e extrai os dados de Pelotas."""
    url = "https://agromet.cpact.embrapa.br/"
    driver.get(url)

    sucesso = acessar_frame_dados(driver)

    if not sucesso:
        driver.switch_to.default_content()
        raise Exception("Não consegui encontrar o frame com os dados da Embrapa.")

    try:
        temperatura = driver.find_element(By.ID, "lblRealV1").text
        umidade = driver.find_element(By.ID, "lblRealV2").text
        sensacao = driver.find_element(By.ID, "lblRealV3").text
        chuva = driver.find_element(By.ID, "lblAcumuladoV1").text

        dados = {
            "cidade": "Pelotas",
            "temperatura": temperatura,
            "umidade": umidade,
            "sensacao": sensacao,
            "chuva": chuva,
        }

        print("🌤️ BOLETIM PELOTAS")
        print("-" * 25)
        print(f"Temperatura: {temperatura}")
        print(f"Umidade: {umidade}")
        print(f"Sensação Térmica: {sensacao}")
        print(f"Chuva Diária: {chuva}")

        return dados

    finally:
        driver.switch_to.default_content()

## 4. Coleta dos dados de Morro Redondo pela Open-Meteo

In [11]:
def extrair_clima_morro_redondo():
    """Busca clima atual de Morro Redondo via Open-Meteo."""
    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=-31.59"
        "&longitude=-52.64"
        "&current=temperature_2m,relative_humidity_2m,apparent_temperature,precipitation"
    )

    try:
        resposta = requests.get(url, timeout=30)
        resposta.raise_for_status()

        payload = resposta.json()
        atual = payload["current"]

        dados_mr = {
            "temperatura_mr": f"{atual['temperature_2m']} °C",
            "umidade_mr": f"{atual['relative_humidity_2m']} %",
            "sensacao_mr": f"{atual['apparent_temperature']} °C",
            "chuva_mr": f"{atual['precipitation']} mm",
        }

        print("🌤️ BOLETIM MORRO REDONDO")
        print("-" * 25)
        print(f"Temperatura: {dados_mr['temperatura_mr']}")
        print(f"Umidade: {dados_mr['umidade_mr']}")
        print(f"Sensação Térmica: {dados_mr['sensacao_mr']}")
        print(f"Chuva: {dados_mr['chuva_mr']}")

        return dados_mr

    except Exception as e:
        raise Exception(f"Erro ao buscar Morro Redondo: {e}")

## 5. Funções para controlar o mapa da REDEMET

Esta versão não tenta acessar diretamente a instância interna do Leaflet. Ela usa:

- `Input.dispatchMouseEvent` do Chrome DevTools Protocol para arrastar.
- A leitura do `Lat/Lon` exibido pela própria página para corrigir a posição aos poucos.

In [12]:
def obter_retangulo_mapa(driver):
    mapa = WebDriverWait(driver, 20).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, ".leaflet-container"))
    )

    rect = driver.execute_script(
        """
        const r = arguments[0].getBoundingClientRect();

        return {
            x: r.x,
            y: r.y,
            width: r.width,
            height: r.height
        };
        """,
        mapa,
    )

    return rect


def mover_mouse_para_centro_do_mapa(driver):
    rect = obter_retangulo_mapa(driver)

    centro_x = rect["x"] + rect["width"] / 2
    centro_y = rect["y"] + rect["height"] / 2

    driver.execute_cdp_cmd(
        "Input.dispatchMouseEvent",
        {
            "type": "mouseMoved",
            "x": centro_x,
            "y": centro_y,
            "button": "none",
        },
    )

    time.sleep(0.3)


def ler_lat_lon_da_tela(driver):
    """Lê o Lat/Lon exibido no canto inferior direito da REDEMET."""
    mover_mouse_para_centro_do_mapa(driver)

    texto = driver.execute_script("return document.body.innerText;")

    match = re.search(
        r"Lat:\s*(-?\d+(?:\.\d+)?)\s*Lon:\s*(-?\d+(?:\.\d+)?)",
        texto,
    )

    if not match:
        raise Exception("Não consegui ler Lat/Lon da tela da REDEMET.")

    lat = float(match.group(1))
    lon = float(match.group(2))

    return lat, lon


def limitar_valor(valor, minimo, maximo):
    return max(min(valor, maximo), minimo)


def arrastar_mapa_cdp(driver, dx, dy):
    """
    Arrasta o mapa usando Chrome DevTools Protocol.
    Isso evita erro de Selenium ActionChains como move target out of bounds.
    """
    rect = obter_retangulo_mapa(driver)

    start_x = rect["x"] + rect["width"] / 2
    start_y = rect["y"] + rect["height"] / 2

    end_x = start_x + dx
    end_y = start_y + dy

    margem = 100

    min_x = rect["x"] + margem
    max_x = rect["x"] + rect["width"] - margem

    min_y = rect["y"] + margem
    max_y = rect["y"] + rect["height"] - margem

    end_x = limitar_valor(end_x, min_x, max_x)
    end_y = limitar_valor(end_y, min_y, max_y)

    driver.execute_cdp_cmd(
        "Input.dispatchMouseEvent",
        {
            "type": "mouseMoved",
            "x": start_x,
            "y": start_y,
            "button": "none",
        },
    )

    driver.execute_cdp_cmd(
        "Input.dispatchMouseEvent",
        {
            "type": "mousePressed",
            "x": start_x,
            "y": start_y,
            "button": "left",
            "clickCount": 1,
        },
    )

    passos = 12

    for i in range(1, passos + 1):
        x = start_x + ((end_x - start_x) * i / passos)
        y = start_y + ((end_y - start_y) * i / passos)

        driver.execute_cdp_cmd(
            "Input.dispatchMouseEvent",
            {
                "type": "mouseMoved",
                "x": x,
                "y": y,
                "button": "left",
            },
        )

        time.sleep(0.03)

    driver.execute_cdp_cmd(
        "Input.dispatchMouseEvent",
        {
            "type": "mouseReleased",
            "x": end_x,
            "y": end_y,
            "button": "left",
            "clickCount": 1,
        },
    )

    time.sleep(0.8)


def centralizar_mapa_por_lat_lon(
    driver,
    lat_alvo,
    lon_alvo,
    tentativas=12,
    tolerancia_lat=0.8,
    tolerancia_lon=0.8,
    pixels_por_grau=35,
):
    """
    Centraliza o mapa usando o Lat/Lon exibido pela própria página como feedback.

    Regras de direção:
    - Para ir mais ao sul: arrastar para cima.
    - Para ir mais ao norte: arrastar para baixo.
    - Para ir mais a leste: arrastar para esquerda.
    - Para ir mais a oeste: arrastar para direita.
    """
    print(f"🎯 Alvo: Lat={lat_alvo}, Lon={lon_alvo}")

    for tentativa in range(1, tentativas + 1):
        lat_atual, lon_atual = ler_lat_lon_da_tela(driver)

        erro_lat = lat_alvo - lat_atual
        erro_lon = lon_alvo - lon_atual

        print(
            f"Tentativa {tentativa}: "
            f"Atual Lat={lat_atual:.2f}, Lon={lon_atual:.2f} | "
            f"Erro Lat={erro_lat:.2f}, Lon={erro_lon:.2f}"
        )

        if abs(erro_lat) <= tolerancia_lat and abs(erro_lon) <= tolerancia_lon:
            print("✅ Mapa centralizado próximo ao alvo.")
            return True

        dy = erro_lat * pixels_por_grau
        dx = -erro_lon * pixels_por_grau

        dx = limitar_valor(dx, -250, 250)
        dy = limitar_valor(dy, -250, 250)

        print(f"Arrastando dx={dx:.0f}, dy={dy:.0f}")
        arrastar_mapa_cdp(driver, dx, dy)

    print("⚠️ Finalizou as tentativas. Pode estar próximo, mas não perfeito.")
    return False


def aplicar_zoom(driver, quantidade=2):
    wait = WebDriverWait(driver, 20)

    btn_zoom = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, ".leaflet-control-zoom-in"))
    )

    for _ in range(quantidade):
        btn_zoom.click()
        time.sleep(0.8)


def ativar_satelite(driver):
    wait = WebDriverWait(driver, 20)

    btn_satelite = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//li[@title='Imagens Satélite']"))
    )

    btn_satelite.click()
    time.sleep(3)


def fechar_menu_lateral(driver):
    wait = WebDriverWait(driver, 20)

    try:
        btn_fechar_menu = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".toggle-sidebar"))
        )
        btn_fechar_menu.click()
        time.sleep(1)
    except Exception as e:
        print(f"⚠️ Não consegui fechar o menu lateral: {e}")

## 6. Captura da imagem de satélite da REDEMET

In [13]:
def capturar_satelite_redemet(driver, lat_alvo=LAT_ALVO, lon_alvo=LON_ALVO):
    print("🌐 Redirecionando o robô para a REDEMET...")

    driver.switch_to.default_content()
    driver.set_window_size(LARGURA_JANELA, ALTURA_JANELA)
    driver.get("https://redemet.decea.mil.br/")

    wait = WebDriverWait(driver, 20)

    wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".leaflet-container"))
    )

    time.sleep(3)

    # Primeiro centraliza no mapa base.
    centralizar_mapa_por_lat_lon(
        driver=driver,
        lat_alvo=lat_alvo,
        lon_alvo=lon_alvo,
        tentativas=12,
        pixels_por_grau=35,
    )

    # Aproxima o zoom.
    aplicar_zoom(driver, quantidade=2)
    time.sleep(1)

    # Recentraliza depois do zoom, porque o zoom muda o enquadramento visual.
    centralizar_mapa_por_lat_lon(
        driver=driver,
        lat_alvo=lat_alvo,
        lon_alvo=lon_alvo,
        tentativas=8,
        pixels_por_grau=55,
        tolerancia_lat=0.4,
        tolerancia_lon=0.4,
    )

    # Ativa camada de satélite.
    ativar_satelite(driver)

    # Fecha menu lateral para melhorar a imagem.
    fechar_menu_lateral(driver)

    print("⏳ Aguardando satélite carregar as nuvens...")
    time.sleep(6)

    foto_binaria = driver.get_screenshot_as_png()

    # Opcional: salva localmente no diretório do notebook.
    with open("satelite_rs.png", "wb") as f:
        f.write(foto_binaria)

    print("📸 Print do satélite capturado com sucesso!")
    print("🖼️ Arquivo salvo como satelite_rs.png")

    return foto_binaria

## 7. Montagem do pacote e envio ao n8n

In [14]:
def montar_pacote_final(boletim_pelotas, boletim_morro_redondo):
    data_hoje = datetime.now().strftime("%d/%m/%Y")

    return {
        "data_hoje": data_hoje,

        "cidade": boletim_pelotas["cidade"],
        "temperatura_pel": boletim_pelotas["temperatura"],
        "umidade_pel": boletim_pelotas["umidade"],
        "sensacao_pel": boletim_pelotas["sensacao"],
        "chuva_pel": boletim_pelotas["chuva"],

        "temperatura_mr": boletim_morro_redondo["temperatura_mr"],
        "umidade_mr": boletim_morro_redondo["umidade_mr"],
        "sensacao_mr": boletim_morro_redondo["sensacao_mr"],
        "chuva_mr": boletim_morro_redondo["chuva_mr"],
    }


def enviar_para_n8n(dados_clima, imagem_binaria, url_webhook=URL_WEBHOOK_N8N):
    print("🚀 Enviando dados e imagem de satélite para o n8n...")

    try:
        arquivos = {
            "imagem_satelite": ("satelite_rs.png", imagem_binaria, "image/png")
        }

        resposta = requests.post(
            url_webhook,
            data=dados_clima,
            files=arquivos,
            timeout=30,
        )

        if resposta.status_code in [200, 201]:
            print("✅ Sucesso! O n8n recebeu o pacote e a imagem.")
        else:
            print(f"⚠️ Erro no n8n. Status: {resposta.status_code}")
            print(resposta.text)

        return resposta

    except Exception as e:
        print(f"❌ Erro de comunicação: {e}")
        return None

## 8. Execução completa

In [16]:
driver = configurar_navegador()

try:
    boletim_pelotas = extrair_dados_clima_pelotas(driver)
    boletim_morro_redondo = extrair_clima_morro_redondo()

    foto_binaria = capturar_satelite_redemet(
        driver=driver,
        lat_alvo=LAT_ALVO,
        lon_alvo=LON_ALVO,
    )

    pacote_final = montar_pacote_final(
        boletim_pelotas=boletim_pelotas,
        boletim_morro_redondo=boletim_morro_redondo,
    )

    resposta_n8n = enviar_para_n8n(
        dados_clima=pacote_final,
        imagem_binaria=foto_binaria,
    )

finally:
    if FECHAR_NAVEGADOR_NO_FINAL:
        driver.quit()

🌤️ BOLETIM PELOTAS
-------------------------
Temperatura: 8.9 °C
Umidade: 90 %
Sensação Térmica: 9.1 °C
Chuva Diária: 0.2 mm
🌤️ BOLETIM MORRO REDONDO
-------------------------
Temperatura: 7.2 °C
Umidade: 92 %
Sensação Térmica: 5.0 °C
Chuva: 0.0 mm
🌐 Redirecionando o robô para a REDEMET...
🎯 Alvo: Lat=-31.67, Lon=-52.48
Tentativa 1: Atual Lat=-15.78, Lon=-55.02 | Erro Lat=-15.89, Lon=2.54
Arrastando dx=-89, dy=-250
Tentativa 2: Atual Lat=-26.76, Lon=-51.11 | Erro Lat=-4.91, Lon=-1.37
Arrastando dx=48, dy=-172
Tentativa 3: Atual Lat=-34.32, Lon=-53.17 | Erro Lat=2.65, Lon=0.69
Arrastando dx=-24, dy=93
Tentativa 4: Atual Lat=-30.23, Lon=-52.08 | Erro Lat=-1.44, Lon=-0.40
Arrastando dx=14, dy=-50
Tentativa 5: Atual Lat=-32.43, Lon=-52.69 | Erro Lat=0.76, Lon=0.21
✅ Mapa centralizado próximo ao alvo.
🎯 Alvo: Lat=-31.67, Lon=-52.48
Tentativa 1: Atual Lat=-32.46, Lon=-52.69 | Erro Lat=0.79, Lon=0.21
Arrastando dx=-12, dy=43
Tentativa 2: Atual Lat=-31.99, Lon=-52.56 | Erro Lat=0.32, Lon=0.08


## 9. Ajustes rápidos

Se o mapa ainda não ficar exatamente na região desejada:

- Aumente `tentativas` na função `centralizar_mapa_por_lat_lon`.
- Ajuste `pixels_por_grau`:
  - Se mexer pouco, aumente.
  - Se passar demais do alvo, diminua.
- Ajuste `LAT_ALVO` e `LON_ALVO` se quiser mirar em outra cidade.

Exemplo para mirar apenas em Pelotas:

```python
LAT_ALVO = -31.7654
LON_ALVO = -52.3376
```

Exemplo para mirar apenas em Morro Redondo:

```python
LAT_ALVO = -31.5886
LON_ALVO = -52.6265
```